# Batch Radiomics Feature Extraction (Normalised Images)
Extract texture and intensity features from all ultrasound images using PyRadiomics
with image-level z-score normalisation enabled (normalizeScale=100, removeOutliers=3).
Each image is paired with its eroded binary mask from the preprocessing pipeline.
Output: a single CSV with one row per study and ~100 feature columns.

In [1]:
import os
import cv2
import pydicom
import numpy as np
import pandas as pd
import SimpleITK as sitk
from radiomics import featureextractor

print("Imports OK")

Imports OK


In [2]:
# paths
raw_data_folder = os.path.join("..", "data", "PANCREAS_2", "PANCREAS_2")
mask_folder = os.path.join("..", "data", "PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED_ERODED_K3_I1", "masks")
manifest_path = os.path.join("..", "data", "PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED_ERODED_K3_I1", "manifest_eroded_CONTOUR_SUBTRACTED_k3_i1.csv")
output_csv_path = os.path.join("reports", "12_radiomics_features_normalised.csv")

# load manifest and check for zero-pixel masks
manifest = pd.read_csv(manifest_path)
zero_mask_rows = manifest[manifest["eroded_pixels"] == 0]
print(f"Manifest loaded: {len(manifest)} studies")
if len(zero_mask_rows) > 0:
    print(f"WARNING: {len(zero_mask_rows)} studies have zero pixels in mask:")
    print(zero_mask_rows["study_id"].tolist())
else:
    print("All masks have nonzero pixels.")

Manifest loaded: 137 studies
All masks have nonzero pixels.


In [3]:
def find_dicom_path(study_id):
    """Find the DICOM file path for a given study ID."""
    patient_folder = os.path.join(raw_data_folder, study_id)
    if not os.path.isdir(patient_folder):
        return None

    # find date subfolder (skip hidden files)
    subfolders = [f for f in os.listdir(patient_folder) if not f.startswith(".")]
    if len(subfolders) == 0:
        return None

    date_folder = os.path.join(patient_folder, subfolders[0])

    # find the DICOM file
    files = [f for f in os.listdir(date_folder) if not f.startswith(".")]
    if len(files) == 0:
        return None

    return os.path.join(date_folder, files[0])

In [4]:
def load_image_and_mask(study_id):
    """Load DICOM as grayscale + mask as binary, return as SimpleITK objects."""
    # load DICOM
    dicom_path = find_dicom_path(study_id)
    if dicom_path is None:
        raise FileNotFoundError(f"No DICOM found for {study_id}")

    ds = pydicom.dcmread(dicom_path)
    pixels = ds.pixel_array

    if len(pixels.shape) == 3:
        gray = cv2.cvtColor(pixels, cv2.COLOR_RGB2GRAY)
    else:
        gray = pixels

    # load mask
    mask_file = study_id + "_mask_eroded_k3_i1.png"
    mask_path = os.path.join(mask_folder, mask_file)
    mask_raw = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask_raw is None:
        raise FileNotFoundError(f"Mask not found: {mask_path}")

    # binarize: 255 -> 1
    mask_binary = np.zeros_like(mask_raw, dtype=np.uint8)
    mask_binary[mask_raw > 0] = 1

    # handle shape mismatch
    if gray.shape != mask_binary.shape:
        mask_binary = cv2.resize(mask_binary, (gray.shape[1], gray.shape[0]),
                                 interpolation=cv2.INTER_NEAREST)

    # cast to int16 -- PyRadiomics texture features can fail on uint8
    gray = gray.astype(np.int16)

    # convert to SimpleITK
    sitk_image = sitk.GetImageFromArray(gray)
    sitk_mask = sitk.GetImageFromArray(mask_binary)

    return sitk_image, sitk_mask

In [5]:
# configure the extractor
# texture + intensity features only, no shape (per supervisor guidance)
# force2D because our images are 2D slices
# normalize=True applies per-ROI z-score normalisation before feature extraction
settings = {
    "force2D": True,
    "force2Ddimension": 0,
    "normalize": True,
    "normalizeScale": 100,
    "removeOutliers": 3,
}
extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
extractor.disableAllFeatures()
extractor.enableFeatureClassByName("firstorder")
extractor.enableFeatureClassByName("glcm")
extractor.enableFeatureClassByName("glrlm")
extractor.enableFeatureClassByName("glszm")
extractor.enableFeatureClassByName("gldm")
extractor.enableFeatureClassByName("ngtdm")

print("Extractor configured (with image normalisation).")
print("Enabled feature classes:", extractor.enabledFeatures)

Extractor configured (with image normalisation).
Enabled feature classes: {'firstorder': [], 'glcm': [], 'glrlm': [], 'glszm': [], 'gldm': [], 'ngtdm': []}


In [6]:
# run extraction on all studies
results = []
errors = []
study_ids = manifest["study_id"].tolist()

for i, study_id in enumerate(study_ids):
    try:
        sitk_image, sitk_mask = load_image_and_mask(study_id)
        features = extractor.execute(sitk_image, sitk_mask)

        # keep only actual features (skip diagnostics_ metadata)
        row = {"study_id": study_id}
        for key, value in features.items():
            if not key.startswith("diagnostics_"):
                row[key] = float(value)

        results.append(row)

    except Exception as e:
        errors.append({"study_id": study_id, "error": str(e)})
        print(f"  ERROR on {study_id}: {e}")

    if (i + 1) % 10 == 0:
        print(f"  processed {i + 1}/{len(study_ids)}")

print(f"\nDone. {len(results)} succeeded, {len(errors)} failed.")
if errors:
    print("Failed studies:")
    for err in errors:
        print(f"  {err['study_id']}: {err['error']}")

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


...

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated



Done. 137 succeeded, 0 failed.


In [7]:
# build dataframe and save
df = pd.DataFrame(results)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")

# list feature classes found
feature_cols = [c for c in df.columns if c != "study_id"]
print(f"Feature columns: {len(feature_cols)}")

# save
os.makedirs("reports", exist_ok=True)
df.to_csv(output_csv_path, index=False)
print(f"Saved to {output_csv_path}")

df.head()

Shape: 137 rows x 94 columns
Feature columns: 93
Saved to reports/12_radiomics_features_normalised.csv


,study_id,original_firstorder_10Percentile,original_firstorder_90Percentile,original_firstorder_Energy,original_firstorder_Entropy,original_firstorder_InterquartileRange,original_firstorder_Kurtosis,original_firstorder_Maximum,original_firstorder_MeanAbsoluteDeviation,original_firstorder_Mean,...,original_gldm_LargeDependenceLowGrayLevelEmphasis,original_gldm_LowGrayLevelEmphasis,original_gldm_SmallDependenceEmphasis,original_gldm_SmallDependenceHighGrayLevelEmphasis,original_gldm_SmallDependenceLowGrayLevelEmphasis,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,01_01,-25.234088,65.896052,2.866846e+07,2.558382,49.070075,2.977687,161.699532,28.512655,17.899627,...,0.635239,0.089393,0.175492,3.596399,0.015510,14.463356,0.001544,8.407328,0.008706,0.070723
1,01_02,-22.400202,64.112915,1.763828e+07,2.470976,44.492460,3.545282,195.118493,26.759747,19.591631,...,0.493037,0.083075,0.247049,5.037135,0.020264,11.018325,0.001611,15.567921,0.008790,0.107989
2,01_03,-16.500631,66.012646,2.233784e+07,2.466571,44.601771,4.058580,181.977251,26.011882,22.901419,...,0.446562,0.072339,0.235105,5.226299,0.016600,12.459464,0.001385,14.798451,0.008707,0.091818
3,01_04,-8.646240,71.288568,2.163365e+07,2.424018,41.109330,3.724079,171.778041,25.150981,31.242682,...,0.385091,0.059338,0.201657,5.000528,0.011957,9.545585,0.001880,10.107560,0.008749,0.086008
4,01_05,60.414817,155.638760,7.683687e+07,2.615661,44.811267,3.953666,273.268336,29.333937,104.960926,...,0.207889,0.039161,0.316017,10.976695,0.012389,4.271825,0.002960,18.607893,0.010522,0.203623
